In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models

In [2]:
from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images   # (1797, 8, 8) real scanned digit images
labels = digits.target   # (1797,) the true digit, 0-9


In [3]:
class RegularizedCNN(nn.Module):
    def __init__(self, num_classes=2, dropout_p=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))   # [B,1,16,16] -> [B,8,8,8]
        x = self.pool(self.relu(self.bn2(self.conv2(x))))   # [B,8,8,8]   -> [B,16,4,4]
        x = self.dropout(x.flatten(start_dim=1))               # -> [B,256]
        x = self.relu(self.fc1(x))                             # -> [B,32]
        x = self.dropout(x)
        return self.fc2(x)                                     # -> [B,num_classes]

In [4]:

model = RegularizedCNN()

split = int(0.8 * len(labels))

# Convert NumPy arrays to PyTorch tensors
images = torch.from_numpy(images).float()
labels = torch.from_numpy(labels).long()

images = images / 16.0

if images.ndim == 3:
    # [N, 8, 8] -> [N, 1, 8, 8]
    images = images.unsqueeze(1)
elif images.ndim == 4:
    # Already [N, 1, 8, 8]
    pass
else:
    raise ValueError(f"Unexpected image shape: {images.shape}")

# # Add channel dimension:
# # (1797, 8, 8) -> (1797, 1, 8, 8)
# images = images.unsqueeze(1)

# 80/20 train-validation split
split = int(0.8 * len(labels))

train_images = images[:split]
train_labels = labels[:split]

val_images = images[split:]
val_labels = labels[split:]

# Create datasets
train_dataset = TensorDataset(train_images, train_labels)


val_dataset = TensorDataset(val_images, val_labels)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)


def random_hflip(batch, p=0.5):
    if torch.rand(1).item() < p:
        return torch.flip(batch, dims=[3])   # flip along the width axis
    return batch

def train(model, loader, epochs=10, lr=1e-3, augment=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()   # IMPORTANT: turns on dropout + batch-norm training-mode stats
    for epoch in range(1, epochs + 1):
        for images, labels in loader:
            if augment:
                images = random_hflip(images)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()


    
# # ... after training:
# model.eval()   # IMPORTANT: turns OFF dropout, freezes batch-norm running stats
# with torch.no_grad():
#     predictions = model(test_images).argmax(dim=1)

In [5]:
train(model, train_loader)

IndexError: Target 8 is out of bounds.

In [12]:
len(labels)

1797

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models

# ==========================================
# 1. BUILD MODEL (Feature Extractor Setup)
# ==========================================
def build_transfer_model(num_classes=2):
    # Load pretrained ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Freeze all existing feature extraction layers
    for param in model.parameters():
        param.requires_grad = False
        
    # Replace the final fully connected layer (512 inputs -> num_classes)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    return model

model = build_transfer_model(num_classes=2)

# ==========================================
# 2. CREATE DUMMY DATA
# ==========================================
# ResNet expects 3-channel RGB images of at least 224x224 pixels
dummy_images = torch.randn(20, 3, 224, 224)
dummy_labels = torch.randint(0, 2, (20,))  # 20 binary labels (0 or 1)

dataset = TensorDataset(dummy_images, dummy_labels)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# ==========================================
# 3. CONFIGURE TRAINING TOOLS
# ==========================================
criterion = nn.CrossEntropyLoss()
# Pass ONLY the parameters of the new head to the optimizer
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

# ==========================================
# 4. TRAINING LOOP (Only trains model.fc)
# ==========================================
model.train()  # Sets training mode (keeps dropout/batchnorm active)

print("--- Training Started ---")
for epoch in range(1, 4):  # Run for 3 epochs
    total_loss = 0.0
    for images, labels in loader:
        optimizer.zero_grad()               # 1. Clear old gradients
        outputs = model(images)             # 2. Forward pass (guess)
        loss = criterion(outputs, labels)   # 3. Check error
        loss.backward()                     # 4. Backward pass (autograd)
        optimizer.step()                    # 5. Update weights (only in model.fc)
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch} | Loss: {total_loss / len(loader):.4f}")
print("--- Training Finished ---\n")

# ==========================================
# 5. INFERENCE / PREDICTION
# ==========================================
model.eval()  # Switch to evaluation mode (turns off dropout, freezes batchnorm)

# Single test image: shape [1, 3, 224, 224]
test_image = torch.randn(1, 3, 224, 224)

with torch.no_grad():  # Disable gradient tracking to save memory
    raw_scores = model(test_image)                   # Raw logits: shape [1, 2]
    predicted_class = raw_scores.argmax(dim=1).item() # Pick the highest score index

class_names = {0: "Class 0", 1: "Class 1"}
print(f"Raw Output Scores: {raw_scores.squeeze().tolist()}")
print(f"Predicted Output : {class_names[predicted_class]}")

ModuleNotFoundError: No module named 'torchvision'